# Train the PBMC SLE model (Colab or Kaggle)

Runs the project pipeline from this repo end to end, then (optionally) uploads the trained model so the
Hugging Face demo can use it.

1. **Colab:** Runtime → Change runtime type → **T4 or L4 GPU**. **Kaggle:** Session options → GPU T4 and Internet On.
2. **Run all.** The first run installs packages and restarts the runtime once; when it reconnects, **Run all** again.
3. It asks for a GitHub token (leave empty if the repo is public) and, only if `UPLOAD = True`, a Hugging Face write token.
4. At the end it downloads `pbmc_outputs.zip` (metrics, attention, DE tables, plots).

Steps: data (download, QC + platelet filter, Perez labels, 5k HVGs) → represent (scVI, CellTypist on Perez labels, UMAP) → endotypes (SLE subtypes) → train (elastic-net baseline vs multi-head attention-MIL, donor-split + cohort-held-out CV) → insights.

In [ ]:
import socket
try:
    socket.create_connection(("pypi.org", 443), timeout=5).close(); print("Internet OK")
except OSError:
    raise RuntimeError("No internet in this notebook session. Kaggle: Session options > Internet On "
                       "(phone-verified account). Or use Colab, where internet is always on.")

In [ ]:
# ---- settings ----
MAX_CELLS_PER_SAMPLE = 500   # fits the 12 GB standard runtime; 1000+ needs High-RAM
UPLOAD = False               # True: push artifacts/ to the Hugging Face model repo the demo reads
MODEL_REPO = "AdamSion74/pbmc-sle-model"
BRANCH = "main"              # or a feature branch name to test it before merging


In [ ]:
import os, sys, getpass, subprocess
os.chdir("/content" if os.path.exists("/content") else "/kaggle/working")
if not os.path.exists("immune-research"):
    token = getpass.getpass("GitHub token with read access to adamsi/immune-research (empty if public): ")
    auth = f"{token}@" if token else ""
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH,
                        f"https://{auth}github.com/adamsi/immune-research.git"], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr.replace(token, "***") if token else r.stderr)
        raise SystemExit("Clone failed. Usual causes: token pasted with a space/typo, token not given access to "
                         "adamsi/immune-research with Contents: Read-only, or BRANCH name wrong. Fix it and re-run this cell.")
os.chdir("immune-research")
# Colab has old pandas/numpy already loaded, so restart once after installing (the clone is kept).
if not os.path.exists(".deps_installed"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    "projects/pbmc_sle_classification/requirements.txt"], check=True)
    open(".deps_installed", "w").close()
    print("Installed. Restarting the runtime now. When it reconnects, click Run all again.")
    os.kill(os.getpid(), 9)
print("Packages ready")

In [ ]:
import yaml
cfg = yaml.safe_load(open("projects/pbmc_sle_classification/configs/default.yaml"))
cfg["data"]["max_cells_per_sample"] = MAX_CELLS_PER_SAMPLE
yaml.safe_dump(cfg, open("projects/pbmc_sle_classification/configs/notebook.yaml", "w"))
CFG = "projects/pbmc_sle_classification/configs/notebook.yaml"
def run(step):
    """Run one pipeline step; stop the notebook if it fails (a `!python` cell would carry on silently)."""
    subprocess.run([sys.executable, "-m", f"projects.pbmc_sle_classification.{step}", "--config", CFG], check=True)


In [ ]:
run("data")


In [ ]:
run("represent")


In [ ]:
run("endotypes")


In [ ]:
run("train")


In [ ]:
run("insights")


In [ ]:
import json
from IPython.display import Image, display
import pandas as pd
R = "projects/pbmc_sle_classification/results"
print(json.dumps(json.load(open(f"{R}/metrics.json")), indent=2))
display(pd.read_csv(f"{R}/attention_by_celltype.csv", index_col=0).head(10))
display(pd.read_csv(f"{R}/composition.csv", index_col=0).head(10))
display(Image(f"{R}/endotypes.png"))
display(Image(f"{R}/umap.png"))

In [ ]:
if UPLOAD:
    from huggingface_hub import HfApi
    api = HfApi(token=getpass.getpass("Hugging Face write token: "))
    api.create_repo(MODEL_REPO, repo_type="model", exist_ok=True)
    api.upload_folder(repo_id=MODEL_REPO, folder_path="projects/pbmc_sle_classification/artifacts",
                      repo_type="model", ignore_patterns=["scvi/*"])
    print(f"Uploaded to https://huggingface.co/{MODEL_REPO}")
print("Commit projects/pbmc_sle_classification/results/metrics.json to show the scores in the demo.")

In [ ]:
# Download the results (metrics.json, attention, DE tables, plots) to your computer
import shutil
shutil.make_archive("/content/pbmc_outputs" if os.path.exists("/content") else "pbmc_outputs", "zip", R)
try:
    from google.colab import files
    files.download("/content/pbmc_outputs.zip")
except ImportError:
    print("Saved pbmc_outputs.zip")
